# 02_feature_engineering.ipynb
- Train a classifier
- Evaluate performance (AUC, precision-recall, etc.)
- Save model using joblib or pickle

# Import & Load Data

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load cleaned data
df = pd.read_csv("cleaned_bill_payments.csv", parse_dates=["bill_due_date", "payment_date"])

df.head()


,user_id,bill_amount,bill_due_date,payment_date,was_late,days_late
0,user_0000,471.57,2024-03-28,2024-03-28,0,0
1,user_0000,64.03,2024-02-01,2024-02-02,1,1
2,user_0000,73.14,2024-02-29,2024-03-01,1,1
3,user_0000,200.41,2024-04-02,2024-04-02,0,0
4,user_0001,83.81,2024-03-28,2024-03-28,0,0


# Feature Engineering

Days before/after due paid

In [7]:
df["days_before_due_paid"] = (df["bill_due_date"] - df["payment_date"]).dt.days

# Temporal features
df["bill_month"] = df["bill_due_date"].dt.month
df["bill_dayofweek"] = df["bill_due_date"].dt.dayofweek

# User-level aggregations

In [8]:
user_features = df.groupby("user_id").agg(
    user_avg_bill_amount=("bill_amount", "mean"),
    user_late_payment_rate=("was_late", "mean"),
    user_bill_count=("bill_amount", "count"),
).reset_index()

# Merge back
df = df.merge(user_features, on="user_id", how="left")

In [9]:
df.head(10)

,user_id,bill_amount,bill_due_date,payment_date,was_late,days_late,days_before_due_paid,bill_month,bill_dayofweek,user_avg_bill_amount,user_late_payment_rate,user_bill_count
0,user_0000,471.57,2024-03-28,2024-03-28,0,0,0,3,3,202.28750,0.500,4
1,user_0000,64.03,2024-02-01,2024-02-02,1,1,-1,2,3,202.28750,0.500,4
2,user_0000,73.14,2024-02-29,2024-03-01,1,1,-1,2,3,202.28750,0.500,4
3,user_0000,200.41,2024-04-02,2024-04-02,0,0,0,4,1,202.28750,0.500,4
4,user_0001,83.81,2024-03-28,2024-03-28,0,0,0,3,3,288.78625,0.375,8
5,user_0001,359.78,2024-02-22,2024-02-22,0,0,0,2,3,288.78625,0.375,8
6,user_0001,418.60,2024-04-05,2024-04-06,1,1,-1,4,4,288.78625,0.375,8
7,user_0001,172.47,2024-03-11,2024-03-11,0,0,0,3,0,288.78625,0.375,8
8,user_0001,91.21,2024-04-20,2024-04-20,0,0,0,4,5,288.78625,0.375,8
9,user_0001,477.61,2024-02-23,2024-02-23,0,0,0,2,4,288.78625,0.375,8


# Select Features & Target

In [10]:
# Feature columns
feature_cols = [
    "bill_amount",
    "days_late",
    "days_before_due_paid",
    "bill_month",
    "bill_dayofweek",
    "user_avg_bill_amount",
    "user_late_payment_rate",
    "user_bill_count"
]

# Target
target_col = "was_late"

X = df[feature_cols]
y = df[target_col]


# Train-Test Split

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")


Training samples: 296
Test samples: 74


# Save Prepared Features

In [12]:
X_train.to_csv("X_train.csv", index=False)
X_test.to_csv("X_test.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("Features and labels saved for training.")


Features and labels saved for training.
